In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

BASE = Path("experiment_v2") if Path("experiment_v2/data").is_dir() else Path(".")

REGION = "13.9_15.2_35.55_36.9"
DAYS = pd.date_range("2008-01-01", "2025-12-31", freq="D")
# The MY ASCAT archive starts in 2019; earlier days are left missing.
WIND_DAYS = pd.date_range("2019-01-01", "2025-12-31", freq="D")

PERIOD = f"{DAYS[0]:%Y-%m-%d}_{DAYS[-1]:%Y-%m-%d}"
WIND_PERIOD = f"{WIND_DAYS[0]:%Y-%m-%d}_{WIND_DAYS[-1]:%Y-%m-%d}"

FILES = {
    "chl": BASE / "data/chl_l3_l4_comparison"
        / f"chl_L3_{PERIOD}_{REGION}.nc",
    "sst": BASE / "data/raw"
        / f"sst_my_202511_{PERIOD}_{REGION}.nc",
    "wind_asc": BASE / "data/raw"
        / f"wind_ascending_my_202311_{WIND_PERIOD}_{REGION}.nc",
    "wind_des": BASE / "data/raw"
        / f"wind_descending_my_202311_{WIND_PERIOD}_{REGION}.nc",
}

WIND_VARIABLES = [
    "eastward_wind",
    "northward_wind",
    "wind_speed",
    "measurement_time",
    "wvc_quality_flag",
]

def load_daily(path, variables, expected_days=DAYS):
    with xr.open_dataset(path) as source:
        ds = source[variables].load()

    ds = ds.rename({
        name: replacement for name, replacement in {"lat" : "latitude", "lon" : "longitude"}.items() if name in ds.dims
    })

    dates = pd.DatetimeIndex(ds.time.values)
    days = dates.normalize()

    if days.has_duplicates:
        raise ValueError(f"Duplicate daily records: {path.name}")
    if not days.sort_values().equals(expected_days):
        raise ValueError(f"Unexpected date coverage: {path.name}")

    ds = ds.assign_coords(time=days)
    ds = ds.sortby("time").sortby("latitude").sortby("longitude")

    # Pad shorter records to the full period; padded days stay missing.
    return ds.reindex(time=DAYS)

chl_source = load_daily(FILES["chl"], ["CHL"])
sst_source = load_daily(FILES["sst"], ["adjusted_sea_surface_temperature"])
asc_source = load_daily(FILES["wind_asc"], WIND_VARIABLES, WIND_DAYS)
des_source = load_daily(FILES["wind_des"], WIND_VARIABLES, WIND_DAYS)


display(pd.DataFrame([
    {
        "Source": name,
        "Days": ds.sizes["time"],
        "Latitude cells": ds.sizes["latitude"],
        "Longitude cells": ds.sizes["longitude"],
        "First day": str(ds.time.values[0])[:10],
        "Last day": str(ds.time.values[-1])[:10],
    }
    for name, ds in [
        ("CHL", chl_source),
        ("SST", sst_source),
        ("Wind ascending", asc_source),
        ("Wind descending", des_source),
    ]
]).set_index("Source"))


,Days,Latitude cells,Longitude cells,First day,Last day
Source,,,,,
CHL,6575,133,101,2008-01-01,2025-12-31
SST,6575,27,26,2008-01-01,2025-12-31
Wind ascending,6575,11,11,2008-01-01,2025-12-31
Wind descending,6575,11,11,2008-01-01,2025-12-31


In [3]:
rows = []

for source, ds in {
    "CHL": chl_source,
    "SST": sst_source,
    "Wind ascending": asc_source,
    "Wind descending": des_source,
}.items():
    for name in ds.data_vars:
        if name in {"measurement_time", "wvc_quality_flag"}:
            continue

        rows.append({
            "Source": source,
            "Variable": name,
            "Units": ds[name].attrs.get("units", ""),
            "Minimum": ds[name].min(skipna=True).item(),
            "Maximum": ds[name].max(skipna=True).item(),
        })

display(pd.DataFrame(rows).round({"Minimum": 4, "Maximum": 4}))

,Source,Variable,Units,Minimum,Maximum
0,CHL,CHL,milligram m^-3,0.00,22927.7402
1,SST,adjusted_sea_surface_temperature,kelvin,285.14,304.1900
2,Wind ascending,eastward_wind,m s-1,-21.06,20.2700
3,Wind ascending,northward_wind,m s-1,-18.71,16.9300
4,Wind ascending,wind_speed,m s-1,0.20,23.1100
5,Wind descending,eastward_wind,m s-1,-34.64,30.5400
6,Wind descending,northward_wind,m s-1,-22.28,16.0800
7,Wind descending,wind_speed,m s-1,0.20,36.8100


In [4]:
chl = chl_source.rename({"CHL": "chl"})

sst = sst_source.rename({
    "adjusted_sea_surface_temperature": "sst_c"
})

sst["sst_c"] = (sst["sst_c"] - 273.15).astype("float32")
sst["sst_c"].attrs = {
    "units": "degree_Celsius",
    "long_name": "Adjusted sea surface foundation temperature",
    "standard_name": "sea_surface_foundation_temperature",
}

display(pd.DataFrame([
    {
        "Variable": name,
        "Units": ds[name].attrs["units"],
        "Minimum": ds[name].min(skipna=True).item(),
        "Maximum": ds[name].max(skipna=True).item(),
    }
    for name, ds in [("chl", chl), ("sst_c", sst)]
]).round({"Minimum": 4, "Maximum": 4}))

,Variable,Units,Minimum,Maximum
0,chl,milligram m^-3,0.00,22927.7402
1,sst_c,degree_Celsius,11.99,31.0400


In [5]:
rows = []

for source, ds in {
    "Ascending": asc_source,
    "Descending": des_source,
}.items():
    flags = ds["wvc_quality_flag"]
    available = flags.notnull() & np.isfinite(ds["wind_speed"])
    total = int(available.sum().item())
    flag_values = flags.fillna(0).astype("int64")

    for mask, meaning in zip(
        flags.attrs["flag_masks"],
        flags.attrs["flag_meanings"].split(),
    ):
        count = int(
            (available & ((flag_values & int(mask)) != 0)).sum().item()
        )

        if count:
            rows.append({
                "Pass": source,
                "Flag": meaning,
                "Pixel-days": count,
                "Observations (%)": 100 * count / total,
            })

display(pd.DataFrame(rows).round({"Observations (%)": 2}))

,Pass,Flag,Pixel-days,Observations (%)
0,Ascending,distance_to_gmf_too_large,1415,1.12
1,Ascending,small_wind_less_than_or_equal_to_3_m_s,29852,23.67
2,Ascending,wind_inversion_not_successful,1415,1.12
3,Ascending,some_portion_of_wvc_is_over_land,105848,83.94
4,Ascending,variational_quality_control_fails,7588,6.02
5,Ascending,knmi_quality_control_fails,12164,9.65
6,Ascending,any_beam_noise_content_above_threshold,10873,8.62
7,Ascending,not_enough_good_sigma0_for_wind_retrieval,18291,14.51
8,Descending,distance_to_gmf_too_large,1273,1.00
9,Descending,small_wind_less_than_or_equal_to_3_m_s,31461,24.76


In [6]:
REJECT_FLAGS = {
    "distance_to_gmf_too_large",
    "wind_inversion_not_successful",
    "variational_quality_control_fails",
    "knmi_quality_control_fails",
    "product_monitoring_event_flag",
    "not_enough_good_sigma0_for_wind_retrieval",
}

wind_masks = {}
rows = []

for name, ds in {"asc": asc_source, "des": des_source}.items():
    flags = ds["wvc_quality_flag"]
    flag_map = dict(zip(
        flags.attrs["flag_meanings"].split(),
        flags.attrs["flag_masks"],
    ))

    bad_mask = sum(int(flag_map[flag]) for flag in REJECT_FLAGS)
    flag_values = flags.fillna(0).astype("int64")

    available = flags.notnull() & np.isfinite(ds["wind_speed"])
    keep = (
        available
        & ((flag_values & bad_mask) == 0)
        & np.isfinite(ds["eastward_wind"])
        & np.isfinite(ds["northward_wind"])
        & (ds["wind_speed"] >= 0)
    )
    wind_masks[name] = keep

    before = int(available.sum().item())
    after = int(keep.sum().item())

    rows.append({
        "Pass": name,
        "Before filtering": before,
        "Retained": after,
        "Retained (%)": 100 * after / before,
        "Maximum retained speed (m/s)":
            ds["wind_speed"].where(keep).max(skipna=True).item(),
    })

display(pd.DataFrame(rows).round({
    "Retained (%)": 2,
    "Maximum retained speed (m/s)": 2,
}))

,Pass,Before filtering,Retained,Retained (%),Maximum retained speed (m/s)
0,asc,126096,95170,75.47,20.60
1,des,127067,98045,77.16,20.03


In [7]:
def prepare_wind(ds, mask, suffix):
    filtered = ds[[
        "eastward_wind",
        "northward_wind",
        "wind_speed",
        "measurement_time",
    ]].where(mask)

    # float32 halves memory; the saved file was already float32.
    for name in ["eastward_wind", "northward_wind", "wind_speed"]:
        filtered[name] = filtered[name].astype("float32")
    
    # Preserve the original flags for diagnostics.
    filtered["wvc_quality_flag"] = ds["wvc_quality_flag"].astype("float32")

    return filtered.rename({
        "eastward_wind": f"wind_u_{suffix}",
        "northward_wind": f"wind_v_{suffix}",
        "wind_speed": f"wind_speed_{suffix}",
        "measurement_time": f"wind_time_{suffix}",
        "wvc_quality_flag": f"wind_qc_{suffix}",
    })

wind_asc = prepare_wind(asc_source, wind_masks["asc"], "asc")
wind_des = prepare_wind(des_source, wind_masks["des"], "des")

print("Ascending variables:", list(wind_asc.data_vars))
print("Descending variables:", list(wind_des.data_vars))

Ascending variables: ['wind_u_asc', 'wind_v_asc', 'wind_speed_asc', 'wind_time_asc', 'wind_qc_asc']
Descending variables: ['wind_u_des', 'wind_v_des', 'wind_speed_des', 'wind_time_des', 'wind_qc_des']


In [8]:
rows = []

for suffix, ds in {"asc": wind_asc, "des": wind_des}.items():
    acquired = ds[f"wind_time_{suffix}"]
    valid = acquired.notnull()

    wrong_day = valid & (acquired.dt.floor("D") != ds.time)
    hours_from_label = (
        (acquired - ds.time) / np.timedelta64(1, "h")
    ).where(valid)

    rows.append({
        "Pass": suffix,
        "Valid acquisition timestamps": int(valid.sum().item()),
        "Different-day timestamps": int(wrong_day.sum().item()),
        "Earliest hour from daily label":
            hours_from_label.min(skipna=True).item(),
        "Latest hour from daily label":
            hours_from_label.max(skipna=True).item(),
    })

display(pd.DataFrame(rows).round(2))

,Pass,Valid acquisition timestamps,Different-day timestamps,Earliest hour from daily label,Latest hour from daily label
0,asc,95170,0,18.92,20.82
1,des,98045,0,7.79,9.65


In [9]:
# CHL has 133 x 101 grid cells, while SST has 27 x 26 larger cells. 
# For each CHL location, the code finds the nearest SST latitude
# and longitude and copies that source cell's temperature
# For example:
# One coarse SST cell:       Several CHL-grid locations:
#   22.5 °C                     22.5   22.5   22.5
#                               22.5   22.5   22.5
# Several neighbouring CHL locations therefore received the same SST
# value. This does not create a finer-resolution temperature
# measurement.
# Missing values stay missing and distance is limited 
# (tolerance=0.51*spacing - hald a source cell along each coordinate) 


def onto_chl_grid(ds):
    aligned = ds

    for coordinate in ["latitude", "longitude"]:
        spacing = float(np.median(np.diff(ds[coordinate].values)))

        aligned = aligned.reindex(
            {coordinate: chl[coordinate].values},
            method="nearest",
            tolerance=0.51 * spacing,
        )

    return aligned

sst_aligned = onto_chl_grid(sst)

coordinates_match = all(
    np.array_equal(sst_aligned[name].values, chl[name].values)
    for name in ["time", "latitude", "longitude"]
)

if not coordinates_match:
    raise ValueError("SST and CHL coordinates do not match.")

print("CHL dimensions:", dict(chl.sizes))
print("SST dimensions:", dict(sst_aligned.sizes))
print("Coordinates match:", coordinates_match)

CHL dimensions: {'time': 6575, 'latitude': 133, 'longitude': 101}
SST dimensions: {'latitude': 133, 'longitude': 101, 'time': 6575}
Coordinates match: True


In [10]:
wind_asc_aligned = onto_chl_grid(wind_asc)
wind_des_aligned = onto_chl_grid(wind_des)

rows = []

for name, ds in {
    "SST": sst_aligned,
    "Wind ascending": wind_asc_aligned,
    "Wind descending": wind_des_aligned,
}.items():
    matches = all(
        np.array_equal(ds[coordinate].values, chl[coordinate].values)
        for coordinate in ["time", "latitude", "longitude"]
    )

    if not matches:
        raise ValueError(f"{name}: coordinates do not match CHL.")

    rows.append({
        "Source": name,
        "Days": ds.sizes["time"],
        "Latitude cells": ds.sizes["latitude"],
        "Longitude cells": ds.sizes["longitude"],
        "Coordinates match CHL": matches,
    })

display(pd.DataFrame(rows).set_index("Source"))

,Days,Latitude cells,Longitude cells,Coordinates match CHL
Source,,,,
SST,6575,133,101,True
Wind ascending,6575,133,101,True
Wind descending,6575,133,101,True


In [11]:
combined = xr.merge(
    [
        chl,
        sst_aligned,
        wind_asc_aligned,
        wind_des_aligned,
    ],
    join="exact",
    compat="no_conflicts",
    combine_attrs="drop",
).transpose("time", "latitude", "longitude")

display(combined)

<xarray.Dataset> Size: 5GB
Dimensions:         (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time            (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude        (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude       (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables:
    chl             (time, latitude, longitude) float32 353MB 0.08791 ... nan
    sst_c           (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_u_asc      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_asc      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_asc  (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_time_asc   (time, latitude, longitude) datetime64[ns] 707MB NaT ... NaT
    wind_qc_asc     (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_u_des      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_des      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_des  (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_time_des   (time, latitude, longitude) datetime64[ns] 707MB NaT ... NaT
    wind_qc_des     (time, latitude, longitude) float32 353MB nan nan ... nan

In [12]:
FEATURES = [
    "chl", "sst_c",
    "wind_u_asc", "wind_v_asc", "wind_speed_asc",
    "wind_u_des", "wind_v_des", "wind_speed_des",
]

rows = []

for name in FEATURES:
    valid = np.isfinite(combined[name])

    rows.append({
        "Variable": name,
        "Valid pixel-days": int(valid.sum().item()),
        "Valid (%)": 100 * valid.mean().item(),
        "Days with observations": int(
            valid.any(dim=["latitude", "longitude"]).sum().item()
        ),
    })

coverage = pd.DataFrame(rows).set_index("Variable")

display(coverage.style.format({
    "Valid pixel-days": "{:,}",
    "Valid (%)": "{:.2f}",
    "Days with observations": "{:,}",
}))

,Valid pixel-days,Valid (%),Days with observations
Variable,,,
chl,"41,236,388",46.69,"5,618"
sst_c,"44,309,034",50.17,"5,805"
wind_u_asc,"10,142,639",11.48,"1,499"
wind_v_asc,"10,142,639",11.48,"1,499"
wind_speed_asc,"10,142,639",11.48,"1,499"
wind_u_des,"10,474,652",11.86,"1,566"
wind_v_des,"10,474,652",11.86,"1,566"
wind_speed_des,"10,474,652",11.86,"1,566"


In [13]:
chl_ok = np.isfinite(combined["chl"])
sst_ok = np.isfinite(combined["sst_c"])
wind_ok = (
    np.isfinite(combined["wind_speed_asc"])
    | np.isfinite(combined["wind_speed_des"])
)

wind_days = int(
    wind_ok.any(dim=["latitude", "longitude"]).sum().item()
)
print(f"Days with either wind pass: {wind_days:,} / {combined.sizes['time']:,}")

chl_count = int(chl_ok.sum().item())
rows = []

for name, mask in {
    "CHL + SST": chl_ok & sst_ok,
    "CHL + either wind pass": chl_ok & wind_ok,
    "CHL + SST + either wind pass": chl_ok & sst_ok & wind_ok,
}.items():
    count = int(mask.sum().item())

    rows.append({
        "Available together": name,
        "Pixel-days": count,
        "Of valid CHL observations (%)": 100 * count / chl_count,
        "Days with overlap": int(
            mask.any(dim=["latitude", "longitude"]).sum().item()
        ),
    })

display(pd.DataFrame(rows).set_index("Available together").style.format({
    "Pixel-days": "{:,}",
    "Of valid CHL observations (%)": "{:.2f}",
    "Days with overlap": "{:,}",
}))

Days with either wind pass: 2,035 / 6,575


,Pixel-days,Of valid CHL observations (%),Days with overlap
Available together,,,
CHL + SST,"28,050,329",68.02,"4,931"
CHL + either wind pass,"9,525,531",23.10,"1,785"
CHL + SST + either wind pass,"7,211,953",17.49,"1,657"


In [14]:
VALIDITY_MASKS = {
    "chl_valid": chl_ok,
    "sst_valid": sst_ok,
    "wind_asc_valid": np.isfinite(combined["wind_speed_asc"]),
    "wind_des_valid": np.isfinite(combined["wind_speed_des"]),
    "wind_any_valid": wind_ok,
}

for name, mask in VALIDITY_MASKS.items():
    combined[name] = mask.astype("uint8")
    combined[name].attrs = {
        "flag_values": np.array([0, 1], dtype="uint8"),
        "flag_meanings": "missing valid",
    }

display(combined[list(VALIDITY_MASKS)])

<xarray.Dataset> Size: 442MB
Dimensions:         (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time            (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude        (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude       (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables:
    chl_valid       (time, latitude, longitude) uint8 88MB 1 1 0 0 0 ... 0 0 0 0
    sst_valid       (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
    wind_asc_valid  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
    wind_des_valid  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
    wind_any_valid  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0

In [15]:
OUT_DIR = BASE / "data/processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "malta_daily_observations.nc"

combined.attrs.update({
    "description": "Daily L3 CHL, SST and ASCAT wind on the CHL grid",
    "spatial_mapping": "Nearest source cell within half-cell tolerance",
    "time_definition": "UTC daily labels; wind acquisition timestamps retained",
    "temporal_gap_filling": "None",
    "source_files": json.dumps({k: str(v) for k, v in FILES.items()}),
})

encoding = {}

for name, variable in combined.variables.items():
    # Remove original packing settings and encoded-range attributes.
    variable.encoding = {}
    variable.attrs.pop("valid_min", None)
    variable.attrs.pop("valid_max", None)

    options = (
        {"zlib": True, "complevel": 4}
        if name in combined.data_vars else {}
    )

    if np.issubdtype(variable.dtype, np.datetime64):
        options.update({
            "units": "seconds since 1970-01-01 00:00:00",
            "dtype": "int64",
            "_FillValue": np.iinfo("int64").min,
        })
    elif np.issubdtype(variable.dtype, np.floating):
        options["dtype"] = "float32"
    elif variable.dtype == np.dtype("uint8"):
        options["_FillValue"] = None

    encoding[name] = options

combined.to_netcdf(
    OUT_PATH,
    engine="netcdf4",
    encoding=encoding,
)

print(f"Saved: {OUT_PATH.resolve()}")
print(f"File size: {OUT_PATH.stat().st_size / 1e6:,.1f} MB")

Saved: /Users/alangatt/git/mcast.pages.chl_prediction/experiment_v2/data/processed/malta_daily_observations.nc
File size: 175.4 MB


In [16]:
print("Number of variables:", len(combined.data_vars))
print("Variable names:", list(combined.data_vars))
display(combined)

Number of variables: 17
Variable names: ['chl', 'sst_c', 'wind_u_asc', 'wind_v_asc', 'wind_speed_asc', 'wind_time_asc', 'wind_qc_asc', 'wind_u_des', 'wind_v_des', 'wind_speed_des', 'wind_time_des', 'wind_qc_des', 'chl_valid', 'sst_valid', 'wind_asc_valid', 'wind_des_valid', 'wind_any_valid']


<xarray.Dataset> Size: 5GB
Dimensions:         (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time            (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude        (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude       (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables: (12/17)
    chl             (time, latitude, longitude) float32 353MB 0.08791 ... nan
    sst_c           (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_u_asc      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_asc      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_asc  (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_time_asc   (time, latitude, longitude) datetime64[ns] 707MB NaT ... NaT
    ...              ...
    wind_qc_des     (time, latitude, longitude) float32 353MB nan nan ... nan
    chl_valid       (time, latitude, longitude) uint8 88MB 1 1 0 0 0 ... 0 0 0 0
    sst_valid       (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
    wind_asc_valid  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
    wind_des_valid  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
    wind_any_valid  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0
Attributes:
    description:           Daily L3 CHL, SST and ASCAT wind on the CHL grid
    spatial_mapping:       Nearest source cell within half-cell tolerance
    time_definition:       UTC daily labels; wind acquisition timestamps reta...
    temporal_gap_filling:  None
    source_files:          {"chl": "data/chl_l3_l4_comparison/chl_L3_2008-01-...

In [17]:
prepared = xr.open_dataset(OUT_PATH)
display(prepared)

<xarray.Dataset> Size: 5GB
Dimensions:         (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time            (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude        (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude       (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables: (12/17)
    chl             (time, latitude, longitude) float32 353MB ...
    sst_c           (time, latitude, longitude) float32 353MB ...
    wind_u_asc      (time, latitude, longitude) float32 353MB ...
    wind_v_asc      (time, latitude, longitude) float32 353MB ...
    wind_speed_asc  (time, latitude, longitude) float32 353MB ...
    wind_time_asc   (time, latitude, longitude) datetime64[ns] 707MB ...
    ...              ...
    wind_qc_des     (time, latitude, longitude) float32 353MB ...
    chl_valid       (time, latitude, longitude) uint8 88MB ...
    sst_valid       (time, latitude, longitude) uint8 88MB ...
    wind_asc_valid  (time, latitude, longitude) uint8 88MB ...
    wind_des_valid  (time, latitude, longitude) uint8 88MB ...
    wind_any_valid  (time, latitude, longitude) uint8 88MB ...
Attributes:
    description:           Daily L3 CHL, SST and ASCAT wind on the CHL grid
    spatial_mapping:       Nearest source cell within half-cell tolerance
    time_definition:       UTC daily labels; wind acquisition timestamps reta...
    temporal_gap_filling:  None
    source_files:          {"chl": "data/chl_l3_l4_comparison/chl_L3_2008-01-...